# Clean Raw Hub

Reads from this notebook's `input/` folder and writes to its `output/` and `reports/` folders (all siblings of this notebook file). Each of the four sections below is self-contained (its own schema constants, `clean_xx()` function, and `df_raw_xx`/`df_cleaned_xx` variables) and can be re-run independently without clobbering another section's results.


## Imports & shared helpers

`find_default_input`, `month_tag_from_filename`, `write_report`, `strip_html`, and `read_semicolon_csv_protecting_backslashes` are identical in shape/logic across the four source notebooks. Here they're defined once, parameterized by `directory`/`prefix`/`filename_re` (and similar) instead of closing over notebook-global constants, and each of the four sections below calls these same functions with its own values.

Schema constants, `clean_xx()` logic, and rename/sort rules genuinely differ per dataset and are kept written out separately in each section rather than hidden behind a generic abstraction.


In [1]:
import csv
import html
import io
import re
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
REPORTS_DIR = NOTEBOOK_DIR / "reports"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


### find_default_input

Looks for a single `{prefix}_yyyy-mm-dd.{ext}` file in a given directory and returns it automatically. If none or several are found, it raises rather than silently guessing which one to use. `ext` defaults to `"csv"`; `HubAssessmentResults` (section 5) passes `ext="xlsx"` since its raw export is a genuine Excel file, not delimited text.


In [2]:
def find_default_input(directory: Path, prefix: str, filename_re: re.Pattern, ext: str = "csv") -> Path:
    matches = sorted(p for p in directory.glob(f"{prefix}_*.{ext}") if filename_re.match(p.name))
    if not matches:
        raise FileNotFoundError(f"No {prefix}_yyyy-mm-dd.{ext} file found in {directory}")
    if len(matches) > 1:
        raise ValueError(
            f"Multiple candidate input files found in {directory}: "
            f"{[m.name for m in matches]}. Pass one explicitly."
        )
    return matches[0]


### month_tag_from_filename

`month_tag` is `yyyymm` for the month *before* the input filename's `yyyy-mm-dd` date suffix (the export date's month minus one). Independent of any raw date data column. `ext` only affects the error message on a non-matching filename; the actual matching is driven entirely by `filename_re`.


In [3]:
def month_tag_from_filename(path: Path, prefix: str, filename_re: re.Pattern, ext: str = "csv") -> str:
    match = filename_re.match(path.name)
    if not match:
        raise ValueError(f"Filename '{path.name}' does not match expected pattern {prefix}_yyyy-mm-dd.{ext}")
    year, month = (int(g) for g in match.groups())
    # month_tag refers to the prior month's data, not the export date's month.
    year, month = (year - 1, 12) if month == 1 else (year, month - 1)
    return f"{year}{month:02d}"


### write_report

Writes the plain-text summary report: input filename, raw row/duplicate counts, `month_tag`, blank/missing-key rows dropped, duplicate rows collapsed, cleaned row/duplicate counts, output filename, then (after three blank lines) `df_cleaned.describe()`. `df_before_dedup` is the cleaned-but-not-yet-deduplicated frame (the output of `clean_xx()`), used so "blank/missing-key rows dropped" keeps its original meaning instead of conflating it with collapsed duplicates. Duplicate counts use pandas' default `duplicated()` (`keep="first"`) — the number of rows that would go away if the dataframe were deduplicated; `Cleaned duplicate rows` should always be `0` after the collapse step below, since every section's `GROUP_COLS` is "every `LS_COLS` field except the summed measures," making post-groupby rows unique by construction. Saved to `reports_dir` as `{prefix}_{month_tag}_report.txt`. Returns `(report_path, report_text)`.


In [4]:
def write_report(
    reports_dir: Path,
    prefix: str,
    month_tag: str,
    input_path: Path,
    df_raw: pd.DataFrame,
    df_before_dedup: pd.DataFrame,
    df_cleaned: pd.DataFrame,
    output_path: Path,
) -> tuple[Path, str]:
    report_lines = [
        f"Input file: {input_path.name}",
        f"Raw row count: {len(df_raw)}",
        f"Raw duplicate rows: {int(df_raw.duplicated().sum())}",
        "=======================================================================",
        f"Month tag: {month_tag}",
        f"Blank/missing-key rows dropped: {len(df_raw) - len(df_before_dedup)}",
        f"Duplicate rows collapsed: {len(df_before_dedup) - len(df_cleaned)}",
        "=======================================================================",
        f"Cleaned row count: {len(df_cleaned)}",
        f"Cleaned duplicate rows: {int(df_cleaned.duplicated().sum())}",
        f"Output file: {output_path.name}",
    ]
    report_text = "\n".join(report_lines) + "\n"
    report_text += "\n\n\n" + df_cleaned.describe().to_string() + "\n"

    reports_dir.mkdir(parents=True, exist_ok=True)
    report_path = reports_dir / f"{prefix}_{month_tag}_report.txt"
    report_path.write_text(report_text, encoding="utf-8")

    return report_path, report_text


### strip_html

Generic HTML cleanup shared by the `HubDailyContent` and `HubDailyEvents` sections: strips tags (including ones with attributes, e.g. `<p style="...">`) via a `<...>` regex, unescapes entities (e.g. `&nbsp;`), and collapses the resulting non-breaking spaces to plain spaces.


In [5]:
HTML_TAG_RE = re.compile(r"<[^>]+>")


def strip_html(value: str) -> str:
    text = HTML_TAG_RE.sub("", value)
    text = html.unescape(text)
    return text.replace("\xa0", " ").strip()


### read_semicolon_csv_protecting_backslashes

Shared by the `HubDailyContent` and `HubDailyEvents` sections, both of which need `engine="python", escapechar="\\"` to parse legitimate backslash-escaped quotes inside HTML attributes (e.g. `<p style=\"...\">`). Left unguarded, `escapechar` strips *every* backslash it precedes, not just ones before a quote — and both raw files contain a real company name, `TBWA\RAAD` (a genuine single backslash, not a CSV escape artifact), which would otherwise be corrupted to `TBWARAAD`. This pre-processes the raw text to double any backslash *not* immediately followed by `"`, so `escapechar` only ever consumes genuine `\"` sequences and every other backslash survives intact.


In [6]:
def read_semicolon_csv_protecting_backslashes(path: Path) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    # Protect literal backslashes that aren't a genuine CSV \" escape by doubling them,
    # so escapechar only ever consumes actual \" sequences below.
    protected_text = re.sub(r'\\(?!")', r"\\\\", raw_text)

    return pd.read_csv(
        io.StringIO(protected_text), sep=";", engine="python", escapechar="\\",
        dtype=str, keep_default_na=False, encoding="utf-8",
    )


## 1. HubDailyContent

Cleans a raw `HubDailyContentData_yyyy-mm-dd.csv` export. Input files use a `...Data` prefix, but output/report files use the shorter `HubDailyContent` prefix (per the notes' own worked example) — see `INPUT_PREFIX_DC` vs `PREFIX_DC` below.


### Schema constants

In [7]:
LS_COLS_DC = [
    "Date", "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode", "Operation",
    "AppInterest", "CategoryName", "ContentTitleLocalLanguage", "ContentTitle", "ContentLanguage",
    "ContentType", "DeviceCategory", "UserType", "Users", "TotalEvents", "UniqueEvents",
    "Stack", "Route", "Topic",
]
LS_STRING_COLS_DC = [
    "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode", "Operation",
    "AppInterest", "CategoryName", "ContentTitleLocalLanguage", "ContentTitle", "ContentLanguage",
    "ContentType", "DeviceCategory", "UserType", "Users", "TotalEvents", "UniqueEvents",
    "Stack", "Route", "Topic",
]
LS_INT_COLS_DC = ["Users", "TotalEvents", "UniqueEvents"]
HTML_COLS_DC = ["ContentTitleLocalLanguage", "ContentTitle", "Stack"]

RENAME_MAP_DC = {"ContentTitle": "ContentTitleLocalLanguage", "ContentTitleEN": "ContentTitle"}

INPUT_PREFIX_DC = "HubDailyContentData"
PREFIX_DC = "HubDailyContent"
FILENAME_RE_DC = re.compile(r"^HubDailyContentData_(\d{4})-(\d{2})-\d{2}\.csv$")

# Duplicate rows are collapsed by grouping on every LS_COLS_DC field except the summed
# measures (Users/TotalEvents/UniqueEvents) and summing those.
GROUP_COLS_DC = [c for c in LS_COLS_DC if c not in LS_INT_COLS_DC]
SUM_COLS_DC = LS_INT_COLS_DC


### Cleaning logic

1. Rename `ContentTitle` → `ContentTitleLocalLanguage` and `ContentTitleEN` → `ContentTitle` (raw local-language/English titles map onto `LS_COLS_DC`'s `ContentTitleLocalLanguage`/`ContentTitle`).
2. Drop rows that are blank across every `LS_COLS_DC` field present in the raw data.
3. Drop rows where `Date` is blank.
4. Reformat `Date` to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds).
5. Reorder/drop columns to match `LS_COLS_DC`.
6. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS_DC` fields (this also includes `Users`/`TotalEvents`/`UniqueEvents`, ahead of the int cast below).
7. Cast `Users`, `TotalEvents`, `UniqueEvents` to integer type.
8. Strip HTML tags/attributes and unescape HTML entities on `ContentTitleLocalLanguage`, `ContentTitle`, `Stack`.


In [8]:
def clean_dc(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns=RENAME_MAP_DC)

    present_ls_cols = [c for c in LS_COLS_DC if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    df = df.loc[~is_blank].copy()

    missing_key = df["Date"].str.strip() == ""
    df = df.loc[~missing_key].copy()

    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    df = df[[c for c in LS_COLS_DC if c in df.columns]]

    for col in LS_STRING_COLS_DC:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_DC:
        df[col] = df[col].astype(int)

    for col in HTML_COLS_DC:
        df[col] = df[col].apply(strip_html)

    return df


### Collapse duplicate rows

Duplicates are not allowed in the final cleaned dataset. Rows that share every `GROUP_COLS_DC` value are collapsed into one row, summing `Users`/`TotalEvents`/`UniqueEvents` as integers.


In [9]:
def collapse_duplicates_dc(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in SUM_COLS_DC:
        df[col] = df[col].astype(int)
    df = df.groupby(GROUP_COLS_DC, as_index=False)[SUM_COLS_DC].sum()
    return df[LS_COLS_DC]


### Configure the input file

Leave `INPUT_FILE_DC` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.


In [10]:
INPUT_FILE_DC = None  # e.g. "input/HubDailyContentData_2026-08-02.csv"

input_path_dc = Path(INPUT_FILE_DC).resolve() if INPUT_FILE_DC else find_default_input(INPUT_DIR, INPUT_PREFIX_DC, FILENAME_RE_DC)
month_tag_dc = month_tag_from_filename(input_path_dc, INPUT_PREFIX_DC, FILENAME_RE_DC)
input_path_dc, month_tag_dc


(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/HUB/input/HubDailyContentData_2026-08-02.csv'),
 '202607')

### Read the raw CSV

Uses the shared `read_semicolon_csv_protecting_backslashes` helper (needs `escapechar` for HTML-attribute quotes, protected against corrupting `TBWA\RAAD`).


In [11]:
df_raw_dc = read_semicolon_csv_protecting_backslashes(input_path_dc)
df_raw_dc.shape


(10102, 30)

### Apply the cleaning steps

In [12]:
df_cleaned_dc = clean_dc(df_raw_dc)
df_cleaned_dc.head()


,Date,CompanyCode,CompanyName,Country,HomeCountry,HomeCountryCode,Operation,AppInterest,CategoryName,ContentTitleLocalLanguage,...,ContentLanguage,ContentType,DeviceCategory,UserType,Users,TotalEvents,UniqueEvents,Stack,Route,Topic
0,2026-07-06 00:00:00.000,ROCHE,ROCHE,Algeria,Algeria,DZ,Lyra Health International Ltd,Relationships,,Gérer ses émotions pendant une rupture,...,fr,Article,Desktop,Returning User,1,1,1,Impactful life changes,Explore,Divorce and break-up
1,2026-07-01 00:00:00.000,TETRAPAK,Tetrapak,Tunisia,Algeria,DZ,Lyra Health International Ltd,"Lifestyle,Relationships",,Comment résoudre les conflits avec les collègues,...,fr,Article,Mobile,Returning User,1,1,1,Interpersonal skills,Explore,Interpersonal skills
2,2026-07-07 00:00:00.000,MOODYS,Moody's Shared Services,Germany,Lithuania,LT,Lyra Health International Ltd,MentalHealth,,Returning to work after a major life change,...,en,Article,Desktop,Returning User,1,1,1,Thriving at work,Explore,Adapting to change
3,2026-07-31 00:00:00.000,BEIERSDORF,BEIERSDORF,Finland,Finland,FI,Lyra Health International Ltd,Lifestyle,Mental Health,How to motivate yourself when you're strugglin...,...,en,Article,Desktop,Returning User,1,1,1,Achieving your goals,Explore,Goal setting
4,2026-07-31 00:00:00.000,BEIERSDORF,BEIERSDORF,Finland,Finland,FI,Lyra Health International Ltd,Relationships,,6 ways to build a stronger team,...,en,Article,Desktop,Returning User,1,1,1,,Explore,"Leadership,Team dynamics,Teambuilding"


### Collapse duplicate rows before saving

`df_before_dedup_dc` is kept so the report below can still report "blank/missing-key rows dropped" against the pre-dedup count, separately from rows collapsed for being duplicates.


In [13]:
df_before_dedup_dc = df_cleaned_dc
df_cleaned_dc = collapse_duplicates_dc(df_before_dedup_dc)
duplicates_collapsed_dc = len(df_before_dedup_dc) - len(df_cleaned_dc)

print(f"Collapsed {duplicates_collapsed_dc} duplicate rows -> {len(df_cleaned_dc)} rows remaining")


Collapsed 113 duplicate rows -> 9985 rows remaining


### Save the cleaned dataset

In [14]:
output_path_dc = OUTPUT_DIR / f"{PREFIX_DC}_{month_tag_dc}_cleaned.csv"
df_cleaned_dc.to_csv(output_path_dc, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_dc)} rows -> {output_path_dc}")


Cleaned 9985 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\output\HubDailyContent_202607_cleaned.csv


### Write summary report

In [15]:
report_path_dc, report_text_dc = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_DC,
    month_tag=month_tag_dc,
    input_path=input_path_dc,
    df_raw=df_raw_dc,
    df_before_dedup=df_before_dedup_dc,
    df_cleaned=df_cleaned_dc,
    output_path=output_path_dc,
)

print(report_text_dc)
print(f"Report written -> {report_path_dc}")


Input file: HubDailyContentData_2026-08-02.csv
Raw row count: 10102
Raw duplicate rows: 73
Month tag: 202607
Blank/missing-key rows dropped: 4
Duplicate rows collapsed: 113
Cleaned row count: 9985
Cleaned duplicate rows: 0
Output file: HubDailyContent_202607_cleaned.csv



             Users  TotalEvents  UniqueEvents
count  9985.000000  9985.000000   9985.000000
mean      1.020831     1.104857      1.040661
std       0.165561     0.445914      0.257011
min       1.000000     1.000000      1.000000
25%       1.000000     1.000000      1.000000
50%       1.000000     1.000000      1.000000
75%       1.000000     1.000000      1.000000
max       6.000000    16.000000     10.000000

Report written -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\reports\HubDailyContent_202607_report.txt


## 2. HubDailyEvents

Cleans a raw `HubDailyEventData_yyyy-mm-dd.csv` export. Like `HubDailyContent`, input files use a `...Data` prefix but output/report files don't — see `INPUT_PREFIX_DE` vs `PREFIX_DE` below.


### Schema constants

In [16]:
LS_COLS_DE = [
    "Date", "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode",
    "Operation", "Feature", "EventAction", "EventCategory", "EventLabel", "EventSection",
    "EventQuestion", "UserType", "Users", "TotalEvents", "UniqueEvents",
    "SessionsWithEvent", "Events/SessionwithEvent",
]
LS_STRING_COLS_DE = [
    "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode", "Operation",
    "EventAction", "EventCategory", "EventLabel", "EventSection", "EventQuestion", "UserType",
]
LS_INT_COLS_DE = ["TotalEvents", "UniqueEvents", "SessionsWithEvent", "Events/SessionwithEvent"]
HTML_COLS_DE = ["EventSection"]

RENAME_MAP_DE = {
    "eventAction": "EventAction",
    "eventCategory": "EventCategory",
    "eventLabel": "EventLabel",
    "eventSection": "EventSection",
    "eventQuestion": "EventQuestion",
    "EventsPerSessionWithEvent": "Events/SessionwithEvent",
}

SORT_COLS_DE = ["Date", "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode"]

INPUT_PREFIX_DE = "HubDailyEventData"
PREFIX_DE = "HubDailyEvents"
FILENAME_RE_DE = re.compile(r"^HubDailyEventData_(\d{4})-(\d{2})-\d{2}\.csv$")

# Duplicate rows are collapsed by grouping on every LS_COLS_DE field except the summed
# measures and summing those. Users isn't in LS_INT_COLS_DE (clean_de never casts it),
# so collapse_duplicates_de casts it defensively before summing.
SUM_COLS_DE = ["Users"] + LS_INT_COLS_DE
GROUP_COLS_DE = [c for c in LS_COLS_DE if c not in SUM_COLS_DE]


### Cleaning logic

1. Rename `eventAction`→`EventAction`, `eventCategory`→`EventCategory`, `eventLabel`→`EventLabel`, `eventSection`→`EventSection`, `eventQuestion`→`EventQuestion`, `EventsPerSessionWithEvent`→`Events/SessionwithEvent`.
2. Drop rows that are blank across every `LS_COLS_DE` field present in the raw data.
3. Drop rows where `Date` is blank.
4. Reformat `Date` to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds).
5. Strip HTML tags/attributes and unescape HTML entities on `EventSection` — before the whitespace pass below, so tag-removal artifacts get collapsed too.
6. Insert a new `Feature` column (always blank — no source data for it), positioned between `Operation` and `EventAction` per `LS_COLS_DE`.
7. Reorder/drop columns to match `LS_COLS_DE`.
8. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS_DE` fields.
9. Cast `TotalEvents`, `UniqueEvents`, `SessionsWithEvent`, `Events/SessionwithEvent` to integer type.
10. Sort ascending by `SORT_COLS_DE`.


In [17]:
def clean_de(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns=RENAME_MAP_DE)

    present_ls_cols = [c for c in LS_COLS_DE if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    df = df.loc[~is_blank].copy()

    missing_key = df["Date"].str.strip() == ""
    df = df.loc[~missing_key].copy()

    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    for col in HTML_COLS_DE:
        df[col] = df[col].apply(strip_html)

    df["Feature"] = ""

    df = df[LS_COLS_DE]

    for col in LS_STRING_COLS_DE:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_DE:
        df[col] = df[col].astype(int)

    df = df.sort_values(by=SORT_COLS_DE, ascending=True).reset_index(drop=True)

    return df


### Collapse duplicate rows

Duplicates are not allowed in the final cleaned dataset. Rows that share every `GROUP_COLS_DE` value are collapsed into one row, summing `SUM_COLS_DE` (`Users`, `TotalEvents`, `UniqueEvents`, `SessionsWithEvent`, `Events/SessionwithEvent`) as integers.


In [18]:
def collapse_duplicates_de(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in SUM_COLS_DE:
        df[col] = df[col].astype(int)
    df = df.groupby(GROUP_COLS_DE, as_index=False)[SUM_COLS_DE].sum()
    return df[LS_COLS_DE]


### Configure the input file

Leave `INPUT_FILE_DE` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.


In [19]:
INPUT_FILE_DE = None  # e.g. "input/HubDailyEventData_2026-08-02.csv"

input_path_de = Path(INPUT_FILE_DE).resolve() if INPUT_FILE_DE else find_default_input(INPUT_DIR, INPUT_PREFIX_DE, FILENAME_RE_DE)
month_tag_de = month_tag_from_filename(input_path_de, INPUT_PREFIX_DE, FILENAME_RE_DE)
input_path_de, month_tag_de


(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/HUB/input/HubDailyEventData_2026-08-02.csv'),
 '202607')

### Read the raw CSV

Uses the shared `read_semicolon_csv_protecting_backslashes` helper (needs `escapechar` for the escaped quotes in `eventLabel`, protected against corrupting `TBWA\RAAD`).


In [20]:
df_raw_de = read_semicolon_csv_protecting_backslashes(input_path_de)
df_raw_de.shape


(73599, 26)

### Apply the cleaning steps

In [21]:
df_cleaned_de = clean_de(df_raw_de)
df_cleaned_de.head()


,Date,CompanyCode,CompanyName,Country,HomeCountry,HomeCountryCode,Operation,Feature,EventAction,EventCategory,EventLabel,EventSection,EventQuestion,UserType,Users,TotalEvents,UniqueEvents,SessionsWithEvent,Events/SessionwithEvent
0,2026-07-01 00:00:00.000,,,Algeria,,,,,link,,,,,Returning User,1,1,1,1,1
1,2026-07-01 00:00:00.000,,,Australia,,,,,link,,,,,Returning User,1,3,2,1,3
2,2026-07-01 00:00:00.000,,,Austria,,,,,link,,,,,Returning User,1,1,1,1,1
3,2026-07-01 00:00:00.000,,,Bulgaria,,,,,link,,,,,Returning User,1,2,2,1,2
4,2026-07-01 00:00:00.000,,,Canada,,,,,link,,,,,Returning User,1,13,10,3,13


### Collapse duplicate rows before saving

`df_before_dedup_de` is kept so the report below can still report "blank/missing-key rows dropped" against the pre-dedup count, separately from rows collapsed for being duplicates.


In [22]:
df_before_dedup_de = df_cleaned_de
df_cleaned_de = collapse_duplicates_de(df_before_dedup_de)
duplicates_collapsed_de = len(df_before_dedup_de) - len(df_cleaned_de)

print(f"Collapsed {duplicates_collapsed_de} duplicate rows -> {len(df_cleaned_de)} rows remaining")


Collapsed 1 duplicate rows -> 73596 rows remaining


### Save the cleaned dataset

In [23]:
output_path_de = OUTPUT_DIR / f"{PREFIX_DE}_{month_tag_de}_cleaned.csv"
df_cleaned_de.to_csv(output_path_de, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_de)} rows -> {output_path_de}")


Cleaned 73596 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\output\HubDailyEvents_202607_cleaned.csv


### Write summary report

In [24]:
report_path_de, report_text_de = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_DE,
    month_tag=month_tag_de,
    input_path=input_path_de,
    df_raw=df_raw_de,
    df_before_dedup=df_before_dedup_de,
    df_cleaned=df_cleaned_de,
    output_path=output_path_de,
)

print(report_text_de)
print(f"Report written -> {report_path_de}")


Input file: HubDailyEventData_2026-08-02.csv
Raw row count: 73599
Raw duplicate rows: 0
Month tag: 202607
Blank/missing-key rows dropped: 2
Duplicate rows collapsed: 1
Cleaned row count: 73596
Cleaned duplicate rows: 0
Output file: HubDailyEvents_202607_cleaned.csv



              Users   TotalEvents  UniqueEvents  SessionsWithEvent  Events/SessionwithEvent
count  73596.000000  73596.000000  73596.000000       73596.000000             73596.000000
mean       1.166504      1.905226      1.407169           1.314854                 1.899560
std        1.105789      5.920023      4.051665           2.692513                 5.909761
min        1.000000      1.000000      1.000000           0.000000                 0.000000
25%        1.000000      1.000000      1.000000           1.000000                 1.000000
50%        1.000000      1.000000      1.000000           1.000000                 1.000000
75%        1.000000      2.000000      1.000000           1.000000                 2.00

## 3. HubDailyUsers

Cleans a raw `HubDailyUsers_yyyy-mm-dd.csv` export. Input and output prefixes are the same here, unlike `HubDailyContent`/`HubDailyEvents`.


### Schema constants

In [25]:
LS_COLS_DU = [
    "Date", "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode",
    "Operation", "DeviceCategory", "UserType", "Users", "Sessions", "SessionDuration",
]
LS_STRING_COLS_DU = [
    "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode",
    "Operation", "DeviceCategory", "UserType",
]
LS_INT_COLS_DU = ["Sessions", "Users"]
SORT_COLS_DU = ["Date", "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode"]

PREFIX_DU = "HubDailyUsers"
FILENAME_RE_DU = re.compile(r"^HubDailyUsers_(\d{4})-(\d{2})-\d{2}\.csv$")

# Duplicate rows are collapsed by grouping on every LS_COLS_DU field except the summed
# measures and summing those. SessionDuration is a hh:mm:ss string, not numeric -- see
# the hms helpers in "Collapse duplicate rows" below.
SUM_COLS_DU = ["Users", "Sessions", "SessionDuration"]
GROUP_COLS_DU = [c for c in LS_COLS_DU if c not in SUM_COLS_DU]


### Cleaning logic

1. Drop the raw `SessionDuration` column and rename `SessionDurationInSeconds` → `SessionDuration` (despite its name, the raw `SessionDurationInSeconds` column actually holds `hh:mm:ss`-formatted strings, not a count of seconds — left as a string, not converted).
2. Drop rows that are blank across every `LS_COLS_DU` field present in the raw data.
3. Drop rows where `Date` is blank.
4. Reformat `Date` to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds).
5. Reorder/drop columns to match `LS_COLS_DU`.
6. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS_DU` fields.
7. Cast `Sessions`, `Users` to integer type.
8. Sort ascending by `SORT_COLS_DU`.


In [26]:
def clean_du(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop(columns=["SessionDuration"]).rename(columns={"SessionDurationInSeconds": "SessionDuration"})

    present_ls_cols = [c for c in LS_COLS_DU if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    df = df.loc[~is_blank].copy()

    missing_key = df["Date"].str.strip() == ""
    df = df.loc[~missing_key].copy()

    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    df = df[[c for c in LS_COLS_DU if c in df.columns]]

    for col in LS_STRING_COLS_DU:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_DU:
        df[col] = df[col].astype(int)

    df = df.sort_values(by=SORT_COLS_DU, ascending=True).reset_index(drop=True)

    return df


### Collapse duplicate rows

Duplicates are not allowed in the final cleaned dataset. Rows that share every `GROUP_COLS_DU` value are collapsed into one row, summing `Users` and `Sessions` as integers. `SessionDuration` is a `hh:mm:ss` string, so it's summed by converting each value to total seconds, adding those, then formatting back to `hh:mm:ss` — hours are left unbounded (not wrapped at 24h) since this is an accumulated duration, not a clock time.


In [27]:
def _hms_to_seconds(value: str) -> int:
    h, m, s = value.split(":")
    return int(h) * 3600 + int(m) * 60 + int(s)


def _seconds_to_hms(total_seconds: int) -> str:
    h, remainder = divmod(total_seconds, 3600)
    m, s = divmod(remainder, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def collapse_duplicates_du(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Users"] = df["Users"].astype(int)
    df["Sessions"] = df["Sessions"].astype(int)
    df = df.groupby(GROUP_COLS_DU, as_index=False).agg({
        "Users": "sum",
        "Sessions": "sum",
        "SessionDuration": lambda s: _seconds_to_hms(sum(_hms_to_seconds(v) for v in s)),
    })
    return df[LS_COLS_DU]


### Configure the input file

Leave `INPUT_FILE_DU` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.


In [28]:
INPUT_FILE_DU = None  # e.g. "input/HubDailyUsers_2026-08-02.csv"

input_path_du = Path(INPUT_FILE_DU).resolve() if INPUT_FILE_DU else find_default_input(INPUT_DIR, PREFIX_DU, FILENAME_RE_DU)
month_tag_du = month_tag_from_filename(input_path_du, PREFIX_DU, FILENAME_RE_DU)
input_path_du, month_tag_du


(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/HUB/input/HubDailyUsers_2026-08-02.csv'),
 '202607')

### Read the raw CSV

Plain read, no `engine`/`escapechar` — this file has no HTML content needing escaped quotes, and empirically those options only corrupt a real company name (`TBWA\RAAD` → `TBWARAAD`).


In [29]:
df_raw_du = pd.read_csv(input_path_du, sep=";", dtype=str, keep_default_na=False, encoding="utf-8")
df_raw_du.shape


(41702, 21)

### Apply the cleaning steps

In [30]:
df_cleaned_du = clean_du(df_raw_du)
df_cleaned_du.head()


,Date,CompanyCode,CompanyName,Country,HomeCountry,HomeCountryCode,Operation,DeviceCategory,UserType,Users,Sessions,SessionDuration
0,2026-07-01 00:00:00.000,,,Algeria,,,,Mobile,Returning User,1,1,00:00:00
1,2026-07-01 00:00:00.000,,,Algeria,,,,Desktop,Returning User,1,1,00:00:00
2,2026-07-01 00:00:00.000,,,Algeria,,null,,Mobile,Returning User,1,1,00:00:00
3,2026-07-01 00:00:00.000,,,Angola,,,,Desktop,Returning User,1,1,00:13:38
4,2026-07-01 00:00:00.000,,,Argentina,,,,Desktop,Returning User,1,1,00:00:00


### Collapse duplicate rows before saving

`df_before_dedup_du` is kept so the report below can still report "blank/missing-key rows dropped" against the pre-dedup count, separately from rows collapsed for being duplicates.


In [31]:
df_before_dedup_du = df_cleaned_du
df_cleaned_du = collapse_duplicates_du(df_before_dedup_du)
duplicates_collapsed_du = len(df_before_dedup_du) - len(df_cleaned_du)

print(f"Collapsed {duplicates_collapsed_du} duplicate rows -> {len(df_cleaned_du)} rows remaining")


Collapsed 12657 duplicate rows -> 29043 rows remaining


### Save the cleaned dataset

In [32]:
output_path_du = OUTPUT_DIR / f"{PREFIX_DU}_{month_tag_du}_cleaned.csv"
df_cleaned_du.to_csv(output_path_du, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_du)} rows -> {output_path_du}")


Cleaned 29043 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\output\HubDailyUsers_202607_cleaned.csv


### Write summary report

In [33]:
report_path_du, report_text_du = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_DU,
    month_tag=month_tag_du,
    input_path=input_path_du,
    df_raw=df_raw_du,
    df_before_dedup=df_before_dedup_du,
    df_cleaned=df_cleaned_du,
    output_path=output_path_du,
)

print(report_text_du)
print(f"Report written -> {report_path_du}")


Input file: HubDailyUsers_2026-08-02.csv
Raw row count: 41702
Raw duplicate rows: 1641
Month tag: 202607
Blank/missing-key rows dropped: 2
Duplicate rows collapsed: 12657
Cleaned row count: 29043
Cleaned duplicate rows: 0
Output file: HubDailyUsers_202607_cleaned.csv



              Users      Sessions
count  29043.000000  29043.000000
mean       2.579279      5.165513
std       10.599625     23.167354
min        1.000000      1.000000
25%        1.000000      1.000000
50%        1.000000      1.000000
75%        2.000000      3.000000
max      463.000000   1053.000000

Report written -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\reports\HubDailyUsers_202607_report.txt


## 4. HubMonthlyUsers

Cleans a raw `HubMonthlyUsers_yyyy-mm-dd.csv` export. The structural odd one out: no `escapechar` (same `TBWA\RAAD` reasoning as `HubDailyUsers`), no HTML columns, no sort step, and an extra `Month` column reformat that the other three sections don't have.


### Schema constants

In [34]:
LS_COLS_MU = [
    "MonthDate", "Month", "CompanyCode", "CompanyName", "Country",
    "HomeCountry", "HomeCountryCode", "Operation",
    "NewUsers", "Users", "Sessions", "Hits",
]
LS_STRING_COLS_MU = [
    "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode", "Operation",
]
LS_INT_COLS_MU = ["NewUsers", "Users", "Sessions", "Hits"]

PREFIX_MU = "HubMonthlyUsers"
FILENAME_RE_MU = re.compile(r"^HubMonthlyUsers_(\d{4})-(\d{2})-\d{2}\.csv$")

# Duplicate rows are collapsed by grouping on every LS_COLS_MU field except the summed
# measures (NewUsers/Users/Sessions/Hits) and summing those.
GROUP_COLS_MU = [c for c in LS_COLS_MU if c not in LS_INT_COLS_MU]
SUM_COLS_MU = LS_INT_COLS_MU


### Cleaning logic

1. Rename `Date` → `MonthDate`.
2. Drop rows that are blank across every `LS_COLS_MU` field present in the raw data.
3. Drop rows where `MonthDate` is blank.
4. Reformat `MonthDate` to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds) and `Month` to `%Y %m`.
5. Reorder/drop columns to match `LS_COLS_MU`.
6. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS_MU` fields.
7. Cast `NewUsers`, `Users`, `Sessions`, `Hits` to integer type.

No sort step here, unlike the other three sections — preserved faithfully, not "fixed" to match.


In [35]:
def clean_mu(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={"Date": "MonthDate"})

    present_ls_cols = [c for c in LS_COLS_MU if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    df = df.loc[~is_blank].copy()

    missing_key = df["MonthDate"].str.strip() == ""
    df = df.loc[~missing_key].copy()

    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["MonthDate"] = pd.to_datetime(df["MonthDate"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]
    df["Month"] = pd.to_datetime(df["Month"], format="%Y-%m").dt.strftime("%Y %m")

    df = df[[c for c in LS_COLS_MU if c in df.columns]]

    for col in LS_STRING_COLS_MU:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_MU:
        df[col] = df[col].astype(int)

    return df


### Collapse duplicate rows

Duplicates are not allowed in the final cleaned dataset. Rows that share every `GROUP_COLS_MU` value are collapsed into one row, summing `NewUsers`/`Users`/`Sessions`/`Hits` as integers.


In [36]:
def collapse_duplicates_mu(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in SUM_COLS_MU:
        df[col] = df[col].astype(int)
    df = df.groupby(GROUP_COLS_MU, as_index=False)[SUM_COLS_MU].sum()
    return df[LS_COLS_MU]


### Configure the input file

Leave `INPUT_FILE_MU` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.


In [37]:
INPUT_FILE_MU = None  # e.g. "input/HubMonthlyUsers_2026-08-02.csv"

input_path_mu = Path(INPUT_FILE_MU).resolve() if INPUT_FILE_MU else find_default_input(INPUT_DIR, PREFIX_MU, FILENAME_RE_MU)
month_tag_mu = month_tag_from_filename(input_path_mu, PREFIX_MU, FILENAME_RE_MU)
input_path_mu, month_tag_mu


(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/HUB/input/HubMonthlyUsers_2026-08-02.csv'),
 '202607')

### Read the raw CSV

Plain read, no `engine`/`escapechar` — same reasoning as `HubDailyUsers`: this file has no HTML content, and those options would corrupt the real company name `TBWA\RAAD`.


In [38]:
df_raw_mu = pd.read_csv(input_path_mu, sep=";", dtype=str, keep_default_na=False, encoding="utf-8")
df_raw_mu.shape


(6183, 20)

### Apply the cleaning steps

In [39]:
df_cleaned_mu = clean_mu(df_raw_mu)
df_cleaned_mu.head()


,MonthDate,Month,CompanyCode,CompanyName,Country,HomeCountry,HomeCountryCode,Operation,NewUsers,Users,Sessions,Hits
0,2026-07-01 00:00:00.000,2026 07,GIAHUB,Gemological Institute of America,Botswana,Botswana,BW,ICAS Botswana,1,1,1,4
1,2026-07-01 00:00:00.000,2026 07,LUCARAHUB,Lucara Botswana,Botswana,Botswana,BW,ICAS Botswana,6,8,8,41
2,2026-07-01 00:00:00.000,2026 07,PULAHUB,Pula Medical Aid Fund,Botswana,Botswana,BW,ICAS Botswana,6,8,11,123
3,2026-07-01 00:00:00.000,2026 07,ICASBWTEST,ICAS Botswana (Test Preview),Botswana,Botswana,BW,ICAS Botswana,2,2,3,27
4,2026-07-01 00:00:00.000,2026 07,ORANGEHUB,Orange Botswana,Botswana,Botswana,BW,ICAS Botswana,9,9,19,236


### Collapse duplicate rows before saving

`df_before_dedup_mu` is kept so the report below can still report "blank/missing-key rows dropped" against the pre-dedup count, separately from rows collapsed for being duplicates.


In [40]:
df_before_dedup_mu = df_cleaned_mu
df_cleaned_mu = collapse_duplicates_mu(df_before_dedup_mu)
duplicates_collapsed_mu = len(df_before_dedup_mu) - len(df_cleaned_mu)

print(f"Collapsed {duplicates_collapsed_mu} duplicate rows -> {len(df_cleaned_mu)} rows remaining")


Collapsed 10 duplicate rows -> 6171 rows remaining


### Save the cleaned dataset

In [41]:
output_path_mu = OUTPUT_DIR / f"{PREFIX_MU}_{month_tag_mu}_cleaned.csv"
df_cleaned_mu.to_csv(output_path_mu, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_mu)} rows -> {output_path_mu}")


Cleaned 6171 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\output\HubMonthlyUsers_202607_cleaned.csv


### Write summary report

In [42]:
report_path_mu, report_text_mu = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_MU,
    month_tag=month_tag_mu,
    input_path=input_path_mu,
    df_raw=df_raw_mu,
    df_before_dedup=df_before_dedup_mu,
    df_cleaned=df_cleaned_mu,
    output_path=output_path_mu,
)

print(report_text_mu)
print(f"Report written -> {report_path_mu}")


Input file: HubMonthlyUsers_2026-08-02.csv
Raw row count: 6183
Raw duplicate rows: 0
Month tag: 202607
Blank/missing-key rows dropped: 2
Duplicate rows collapsed: 10
Cleaned row count: 6171
Cleaned duplicate rows: 0
Output file: HubMonthlyUsers_202607_cleaned.csv



          NewUsers        Users      Sessions          Hits
count  6171.000000  6171.000000   6171.000000   6171.000000
mean      7.352941     8.898234     24.237401     63.422946
std     103.521488   122.471859    273.691851    530.360765
min       0.000000     1.000000      1.000000      1.000000
25%       1.000000     1.000000      1.000000      2.000000
50%       1.000000     1.000000      2.000000      6.000000
75%       2.000000     3.000000      5.000000     22.000000
max    6520.000000  7890.000000  10324.000000  23512.000000

Report written -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\reports\HubMonthlyUsers_202607_report.txt


## 5. HubAssessmentResults

Cleans a raw `HubAssessmentResults_yyyy-mm-dd.xlsx` export (Sheet1) — the structural odd one out: the only section reading a genuine Excel file rather than a CSV (`pd.read_excel`, via the shared `find_default_input`/`month_tag_from_filename` helpers' `ext="xlsx"` parameter), no `escapechar`/backslash-protection needed since Excel cells aren't delimiter-parsed at all (there's no escape sequence to protect, and empirically zero literal backslashes exist in the data anyway), and no duplicate-collapsing step (per the notes) — `write_report` below is called with `df_cleaned_ar` for both dedup-related parameters, so `Duplicate rows collapsed` reports as `0` while the raw/cleaned duplicate counts still surface any that exist.


### Schema constants

In [43]:
LS_COLS_AR = [
    "Date", "CompanyCode", "CompanyName", "CurrentCountry", "HomeCountry",
    "Operation", "AssessmentType", "SectionName", "Result",
]
LS_STRING_COLS_AR = [
    "CompanyCode", "CompanyName", "CurrentCountry", "HomeCountry",
    "Operation", "AssessmentType", "SectionName", "Result",
]
LS_INT_COLS_AR = []
HTML_COLS_AR = ["SectionName"]
SORT_COLS_AR = ["Date", "CompanyCode", "CompanyName", "CurrentCountry", "HomeCountry", "Operation"]

PREFIX_AR = "HubAssessmentResults"
FILENAME_RE_AR = re.compile(r"^HubAssessmentResults_(\d{4})-(\d{2})-\d{2}\.xlsx$")


### Cleaning logic

1. Drop rows that are blank across every `LS_COLS_AR` field present in the raw data.
2. Drop rows where `Date` is blank.
3. Reformat `Date` from its raw `yyyy-mm-dd hh:mm:ss.fffffff +00:00` shape to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds) — this dataset's raw timestamp already carries full time-of-day precision plus a UTC offset, unlike the plain `yyyy-mm-dd` dates the other four sections parse.
4. Strip HTML tags/entities from `SectionName` (shared `strip_html`).
5. Lowercase `Result`.
6. Reorder/drop columns to match `LS_COLS_AR`.
7. Trim and collapse whitespace on the `LS_STRING_COLS_AR` fields.
8. Cast `LS_INT_COLS_AR` to integer type (no-op — this dataset has none).
9. Sort ascending by `SORT_COLS_AR`.


In [44]:
def clean_ar(df: pd.DataFrame) -> pd.DataFrame:
    present_ls_cols = [c for c in LS_COLS_AR if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    df = df.loc[~is_blank].copy()

    missing_key = df["Date"].str.strip() == ""
    df = df.loc[~missing_key].copy()

    # The raw timestamp carries 7-digit fractional seconds and a UTC offset (always
    # "+00:00"); %f zero-pads/truncates to 6-digit microseconds, and slicing off the
    # last 3 leaves milliseconds. Dropping the offset from the output format loses no
    # information since every row is already UTC.
    df["Date"] = (
        pd.to_datetime(df["Date"], format="%Y-%m-%d %H:%M:%S.%f %z")
        .dt.strftime("%Y-%m-%d %H:%M:%S.%f")
        .str[:-3]
    )

    for col in HTML_COLS_AR:
        df[col] = df[col].apply(strip_html)

    df["Result"] = df["Result"].str.lower()

    df = df[[c for c in LS_COLS_AR if c in df.columns]]

    for col in LS_STRING_COLS_AR:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_AR:
        df[col] = df[col].astype(int)

    df = df.sort_values(by=SORT_COLS_AR, ascending=True).reset_index(drop=True)

    return df


### Configure the input file

Leave `INPUT_FILE_AR` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.


In [45]:
INPUT_FILE_AR = None  # e.g. "input/HubAssessmentResults_2026-08-02.xlsx"

input_path_ar = Path(INPUT_FILE_AR).resolve() if INPUT_FILE_AR else find_default_input(INPUT_DIR, PREFIX_AR, FILENAME_RE_AR, ext="xlsx")
month_tag_ar = month_tag_from_filename(input_path_ar, PREFIX_AR, FILENAME_RE_AR, ext="xlsx")
input_path_ar, month_tag_ar


(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/HUB/input/HubAssessmentResults_2026-08-02.xlsx'),
 '202607')

### Read the raw Excel file

Read everything as strings (`dtype=str`, `keep_default_na=False`) from `Sheet1`, so blank fields and literal `"NULL"` text pass through unchanged instead of being coerced or turned into `NaN`. No `read_semicolon_csv_protecting_backslashes`/`escapechar` handling here — `pd.read_excel` reads cell values directly rather than tokenizing a delimited text stream, so there's no escape sequence to protect in the first place.


In [46]:
df_raw_ar = pd.read_excel(input_path_ar, sheet_name="Sheet1", dtype=str, keep_default_na=False)
df_raw_ar.shape


(8960, 9)

### Apply the cleaning steps

In [47]:
df_cleaned_ar = clean_ar(df_raw_ar)
df_cleaned_ar.head()


,Date,CompanyCode,CompanyName,CurrentCountry,HomeCountry,Operation,AssessmentType,SectionName,Result
0,2026-07-01 00:21:23.916,AROLLAHUB,Arolla,France,France,Lyra France SASU,checkIn,NULL,sad
1,2026-07-01 00:24:03.369,FERRERO,Ferrero,United States of America,United States of America,Lyra Health International Ltd,checkIn,NULL,content
2,2026-07-01 00:46:11.677,SANDVIK,Sandvik Mining and Construction SEA Pte Ltd.,Hong Kong,Philippines,Lyra Health International Ltd,checkIn,NULL,happy
3,2026-07-01 01:06:12.517,BPWELLBEINGUSA,BP,United States of America,United States of America,Lyra Health International Ltd,checkIn,NULL,content
4,2026-07-01 01:43:03.624,KOCH,Koch,Mexico,Mexico,Lyra Health International Ltd,checkIn,NULL,happy


### Save the cleaned dataset

In [48]:
output_path_ar = OUTPUT_DIR / f"{PREFIX_AR}_{month_tag_ar}_cleaned.csv"
df_cleaned_ar.to_csv(output_path_ar, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_ar)} rows -> {output_path_ar}")


Cleaned 8960 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\output\HubAssessmentResults_202607_cleaned.csv


### Write summary report

In [49]:
report_path_ar, report_text_ar = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_AR,
    month_tag=month_tag_ar,
    input_path=input_path_ar,
    df_raw=df_raw_ar,
    df_before_dedup=df_cleaned_ar,
    df_cleaned=df_cleaned_ar,
    output_path=output_path_ar,
)

print(report_text_ar)
print(f"Report written -> {report_path_ar}")


Input file: HubAssessmentResults_2026-08-02.xlsx
Raw row count: 8960
Raw duplicate rows: 0
Month tag: 202607
Blank/missing-key rows dropped: 0
Duplicate rows collapsed: 0
Cleaned row count: 8960
Cleaned duplicate rows: 0
Output file: HubAssessmentResults_202607_cleaned.csv



                           Date CompanyCode       CompanyName CurrentCountry   HomeCountry                      Operation AssessmentType SectionName   Result
count                      8960        8960              8960           8960          8960                           8960           8960        8960     8960
unique                     8959         826               825            114           117                             24              2           7       10
top     2026-07-02 11:47:59.916    MAFXLYRA  Majid Al Futtaim   South Africa  South Africa  Lyra Health International Ltd        checkIn        NULL  content
freq                          2         425               425           2876          2809 

## Summary

Convenience recap of everything produced by this run.


In [50]:
print("Cleaned outputs:")
for label, out_path, rpt_path in [
    ("DailyContent", output_path_dc, report_path_dc),
    ("DailyEvents",  output_path_de, report_path_de),
    ("DailyUsers",   output_path_du, report_path_du),
    ("MonthlyUsers", output_path_mu, report_path_mu),
    ("AssessmentResults", output_path_ar, report_path_ar),
]:
    print(f"  {label:14s} -> {out_path.name}  (report: {rpt_path.name})")


Cleaned outputs:
  DailyContent   -> HubDailyContent_202607_cleaned.csv  (report: HubDailyContent_202607_report.txt)
  DailyEvents    -> HubDailyEvents_202607_cleaned.csv  (report: HubDailyEvents_202607_report.txt)
  DailyUsers     -> HubDailyUsers_202607_cleaned.csv  (report: HubDailyUsers_202607_report.txt)
  MonthlyUsers   -> HubMonthlyUsers_202607_cleaned.csv  (report: HubMonthlyUsers_202607_report.txt)
  AssessmentResults -> HubAssessmentResults_202607_cleaned.csv  (report: HubAssessmentResults_202607_report.txt)
